# Teaching transformers how to write like Shakespeare

### Imports

In [1]:
import sys
import torch 
from tqdm import tqdm
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve() / "src"))
from transformers.word_embedding import word_embedding
from transformers.utils import estimate_loss, get_batch
from transformers.transformer import GPTLanguageModel

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/juanfrancisco/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/juanfrancisco/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
input_file_path = "/Users/juanfrancisco/Desktop/Transformers/data/tinyshakespeare/"

# # download the tiny shakespeare dataset
# if not os.path.exists(input_file_path + "input.txt"):
#     data_url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
#     with open(input_file_path + "input.txt", 'w', encoding='utf-8') as f:
#         f.write(requests.get(data_url).text)

with open(input_file_path + "input.txt", 'r', encoding='utf-8') as f:
    text = f.read()

print(text[:100])
print(f"length of dataset in characters: {len(text)}")

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You
length of dataset in characters: 1115394


### Word embedding

In [3]:
text = text[:1000]

embedding_model = word_embedding(embedding_type="gpt2")
tokens = embedding_model.encode(text)
n_vocab = embedding_model.tokenizer.n_vocab
embedding = embedding_model.embed(text)
# embedding = torch.nn.Embedding(n_vocab, n_vocab)
# embedding.load_state_dict(torch.load(os.path.join(input_file_path, 'embedding.pt')))
data = torch.tensor(tokens, dtype=torch.long)

# Split training and validation dataset
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

In [4]:
print(f"train has {len(train_data):,} tokens")
print(f"val has {len(val_data):,} tokens")

train has 256 tokens
val has 29 tokens


In [5]:
# # export files
# train_ids = np.array(train_data, dtype=np.uint16)
# val_ids = np.array(val_data, dtype=np.uint16)
# train_ids.tofile(os.path.join(input_file_path, 'train.bin'))
# val_ids.tofile(os.path.join(input_file_path, 'val.bin'))
# torch.save(embedding.state_dict(), os.path.join(input_file_path, 'embedding.pt'))

Split data

In [6]:
block_size = 8
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

when input is tensor([5962]) the target: 22307
when input is tensor([ 5962, 22307]) the target: 25
when input is tensor([ 5962, 22307,    25]) the target: 198
when input is tensor([ 5962, 22307,    25,   198]) the target: 8421
when input is tensor([ 5962, 22307,    25,   198,  8421]) the target: 356
when input is tensor([ 5962, 22307,    25,   198,  8421,   356]) the target: 5120
when input is tensor([ 5962, 22307,    25,   198,  8421,   356,  5120]) the target: 597
when input is tensor([ 5962, 22307,    25,   198,  8421,   356,  5120,   597]) the target: 2252


In [7]:
torch.manual_seed(1337)
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?

xb, yb = get_batch('train', train_data, val_data, block_size, batch_size)
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

inputs:
torch.Size([4, 8])
tensor([[ 2949,   517,  3375,   319,   470,    26,  1309,   340],
        [  475,   262, 48713,   414,    11,   981,   340,   547],
        [  198,   198,  5962, 22307,    25,   198,  5962,    11],
        [15593,    30,   198,   198,  3237,    25,   198,  2949]])
targets:
torch.Size([4, 8])
tensor([[  517,  3375,   319,   470,    26,  1309,   340,   307],
        [  262, 48713,   414,    11,   981,   340,   547,   198],
        [  198,  5962, 22307,    25,   198,  5962,    11,   345],
        [   30,   198,   198,  3237,    25,   198,  2949,   517]])


### Training

In [8]:
# # hyperparameters GPU
# batch_size = 64 
# block_size = 256
# max_iters = 5000
# eval_interval = 500
# vocab_size = embedding.num_embeddings
# vocab_dim = 32
# num_heads = 6
# n_layer = 6
# dropout = 0.2
# learning_rate = 3e-4

# hyperparameters for CPU
batch_size = 8  # number of parallel training sequences
block_size = 8    # number of tokens per batch
vocab_size = embedding.num_embeddings
vocab_dim = 32   # dimension of each token
num_heads = 6   # set of Q,K,V matrices during self attention
n_layer = 6     # number of blocks
dropout = 0.2  # percentage of neurons to drop during training, helps prevent overfitting
learning_rate = 3e-4
max_iters = 10  # training iterarions/epochs
eval_interval = 1   

In [10]:
# create a PyTorch optimizer
model = GPTLanguageModel(vocab_size, vocab_dim, block_size, num_heads, n_layer, dropout)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in tqdm(range(max_iters)):

    # sample a batch of data
    xb, yb = get_batch('train', train_data, val_data, block_size, batch_size)

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss(model, train_data, val_data, block_size, batch_size, max_iters)
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)

    #update parameters
    loss.backward()
    optimizer.step()

 10%|█         | 1/10 [00:00<00:02,  4.29it/s]

step 0: train loss 10.9224, val loss 11.0532


 30%|███       | 3/10 [00:00<00:00,  7.14it/s]

step 1: train loss 10.9165, val loss 11.0382
step 2: train loss 10.9027, val loss 11.0405


 50%|█████     | 5/10 [00:00<00:00,  7.97it/s]

...

step 5: train loss 10.8297, val loss 11.0287
step 6: train loss 10.8063, val loss 10.9737


 90%|█████████ | 9/10 [00:01<00:00,  8.29it/s]

step 7: train loss 10.8029, val loss 11.0068
step 8: train loss 10.7519, val loss 11.0286


100%|██████████| 10/10 [00:01<00:00,  7.52it/s]

step 9: train loss 10.7884, val loss 10.9667


In [11]:
generated_tokens = model.generate(idx = torch.zeros((1, 8), dtype=torch.long), max_new_tokens=50)
decoded_text = embedding_model.decode(generated_tokens[0].tolist())
print(decoded_text)

!!!!!!!!__ crawlingarijuanaï catchingEnhanced Transition Puerto Pumpkin disruptres generous takedown vents loophole Participantsredited Reduced conjectureitement Drivercut debated ENTERArg educating erosion 361evil APIsSkip scripting Spyotropribution tut xx NPCs prioritize pleaanti Spanish par discriminatory bulliedVM introduce cowardly updates rocking
